In [2]:

import os

print("Available Kaggle inputs:")

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))


Available Kaggle inputs:
/kaggle/input/datasets/tazeenurrehman/horizontal-beat-nystagmus-bppv-viewer-video-3/Untitled design.mp4
/kaggle/input/datasets/tazeenurrehman/eye-gaze-server-files/eye_server/README.md
/kaggle/input/datasets/tazeenurrehman/eye-gaze-server-files/eye_server/requirements.txt
/kaggle/input/datasets/tazeenurrehman/eye-gaze-server-files/eye_server/pipeline/__init__.py
/kaggle/input/datasets/tazeenurrehman/eye-gaze-server-files/eye_server/pipeline/config.py
/kaggle/input/datasets/tazeenurrehman/eye-gaze-server-files/eye_server/pipeline/ellseg_stage.py
/kaggle/input/datasets/tazeenurrehman/eye-gaze-server-files/eye_server/pipeline/eye_crop.py
/kaggle/input/datasets/tazeenurrehman/eye-gaze-server-files/eye_server/pipeline/gaze_signal.py
/kaggle/input/datasets/tazeenurrehman/eye-gaze-server-files/eye_server/pipeline/mock.py
/kaggle/input/datasets/tazeenurrehman/eye-gaze-server-files/eye_server/pipeline/nystagmus_stage.py
/kaggle/input/datasets/tazeenurrehman/eye-gaze-ser

In [3]:
import os
import shutil

source = "/kaggle/input/datasets/tazeenurrehman/eye-gaze-server-files/eye_server"
destination = "/kaggle/working/eye_server"

# Copy the package into the writable working directory
if os.path.exists(destination):
    shutil.rmtree(destination)

shutil.copytree(source, destination)

print("Package copied successfully!")
print("\nPackage contents:")
for item in os.listdir(destination):
    print("-", item)

print("\n\nContents of requirements.txt:")
with open(os.path.join(destination, "requirements.txt"), "r") as f:
    print(f.read())

Package copied successfully!

Package contents:
- server
- requirements.txt
- pipeline
- README.md
- tests


Contents of requirements.txt:
fastapi
uvicorn
python-multipart
numpy
pandas
scipy
opencv-python-headless
PyWavelets
imageio-ffmpeg
# EllSeg stage (Kaggle already has torch with a GPU):
torch
torchvision
scikit-image
scikit-learn
tqdm
matplotlib



In [4]:
import importlib.util
import importlib.metadata
import torch

packages = [
    "fastapi",
    "uvicorn",
    "python-multipart",
    "numpy",
    "pandas",
    "scipy",
    "opencv-python-headless",
    "PyWavelets",
    "imageio-ffmpeg",
    "torch",
    "torchvision",
    "scikit-image",
    "scikit-learn",
    "tqdm",
    "matplotlib"
]

print("PACKAGE AVAILABILITY\n")

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package}: installed ({version})")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nPYTORCH / GPU CHECK")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PACKAGE AVAILABILITY

fastapi: installed (0.141.1)
uvicorn: installed (0.54.0)
python-multipart: installed (0.0.32)
numpy: installed (2.1.3)
pandas: installed (2.3.3)
scipy: installed (1.16.3)
opencv-python-headless: installed (4.14.0.94)
PyWavelets: installed (1.10.0)
imageio-ffmpeg: installed (0.6.0)
torch: installed (2.11.0+cu130)
torchvision: installed (0.26.0+cu130)
scikit-image: installed (0.25.2)
scikit-learn: installed (1.6.1)
tqdm: installed (4.67.3)
matplotlib: installed (3.10.0)

PYTORCH / GPU CHECK
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [5]:
import os
import sys
import time
from pathlib import Path

os.chdir("/kaggle/working/eye_server")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from pipeline.ellseg_stage import EllSegRunner

runner = EllSegRunner()

print("Loading EllSeg...")
start = time.time()

runner.load()

elapsed = time.time() - start

print("\nEllSeg loaded successfully!")
print("Device:", runner.device)
print("Loading time:", round(elapsed, 2), "seconds")

weight_path = Path("EllSeg/weights/all.git_ok")
print("Checkpoint exists:", weight_path.exists())

if weight_path.exists():
    print("Checkpoint size:", round(weight_path.stat().st_size / (1024 * 1024), 2), "MB")

Loading EllSeg...


/kaggle/working/eye_server/pipeline/ellseg_stage.py:21: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, n):
Cloning into 'EllSeg'...



EllSeg loaded successfully!
Device: cuda
Loading time: 8.64 seconds
Checkpoint exists: True
Checkpoint size: 9.97 MB


In [6]:
import os
import time
from pipeline.run import analyze

video_path = "/kaggle/input/datasets/tazeenurrehman/horizontal-beat-bppv-viewer-video-4/Untitled design (1).mp4"

print("Starting real-video analysis...")
start_time = time.time()

result = analyze(video_path, runner)

elapsed = time.time() - start_time

print("\n" + "=" * 50)
print("ANALYSIS COMPLETED")
print("=" * 50)

print("\nProcessing time:", round(elapsed, 2), "seconds")

print("\nHEADLINE:")
print(result.get("headline"))

print("\nOVERALL:")
print(result.get("overall"))

print("\nWARNINGS:")
for warning in result.get("warnings", []):
    print("-", warning)

print("\nMETADATA:")
meta = result.get("meta", {})
for key, value in meta.items():
    print(f"{key}: {value}")

print("\nNYSTAGMUS RESULTS:")
for axis in ["horizontal", "vertical"]:
    data = result.get("nystagmus", {}).get(axis, {})
    print(f"\n{axis.upper()}:")
    for key in [
        "label",
        "accepted_beats",
        "median_spv",
        "direction_consistency",
        "rejections",
        "notes"
    ]:
        print(f"{key}: {data.get(key)}")

print("\nSIGNAL LENGTHS:")
for key in [
    "timestamps",
    "horizontal_angle",
    "vertical_angle",
    "velocity_h",
    "velocity_v",
    "spv_h",
    "spv_v",
    "blink"
]:
    print(f"{key}: {len(result.get(key, []))}")

Starting real-video analysis...


Cloning into 'DeepVOG'...



ANALYSIS COMPLETED

Processing time: 314.56 seconds

HEADLINE:
Horizontal: Left-beating

OVERALL:
nystagmus_detected

WARNINGS:

METADATA:
fps: 60.0
unit: deg
angle_mode: eyeball_model
valid_frames: 7271
eyeball_radius_mm: 11.985554867570572
crop: {'x0': 278, 'y0': 184, 'w': 664, 'h': 498}
n_frames: 7571
duration_s: 126.167
blink_frames: 126
timings_s: {'read_video': 0.48, 'find_eye': 10.23, 'crop': 23.72, 'ellseg': 213.58, 'eyeball_model': 65.87, 'signal': 0.06, 'nystagmus': 0.37}

NYSTAGMUS RESULTS:

HORIZONTAL:
label: Left-beating
accepted_beats: 70
median_spv: 8.119510209775312
direction_consistency: 0.7571428571428571
rejections: {'no_direction_reversal': 86, 'too_few_samples': 446, 'r2_too_low': 77, 'spv_too_small': 27, 'accepted': 70, 'slow_duration_out_of_range': 3, 'touches_blink': 2}
notes: []

VERTICAL:
label: Inconclusive (conflicting patterns)
accepted_beats: 19
median_spv: 2.297090363129613
direction_consistency: 0.5789473684210527
rejections: {'too_few_samples': 347, 's